In [90]:
# Install requests library for making API calls
!pip install requests pandas matplotlib seaborn --quiet
print('Libraries ready!')

Libraries ready!


In [91]:
# requests: used to send HTTP requests to the Hugging Face API
import requests

# json: used to convert Python objects to JSON format for API
import json

# pandas: for organizing results in a table
import pandas as pd

# For visualization
import matplotlib.pyplot as plt
import seaborn as sns

# time: to add small delays between API calls (avoid rate limits)
import time

import warnings
warnings.filterwarnings('ignore')

print('All libraries imported!')

All libraries imported!


In [92]:
GROQ_API_KEY = 'your_groq_api_key_here'  # paste gsk_... key here

GROQ_URL = 'https://api.groq.com/openai/v1/chat/completions'

HEADERS = {
    'Authorization': f'Bearer {GROQ_API_KEY}',
    'Content-Type': 'application/json'
}

In [93]:
# These are the categories we want the AI to choose from
# In a real company, these would match their support departments

AVAILABLE_TAGS = [
    'Billing Issue',        # payment, invoice, charges problems
    'Technical Problem',    # software/hardware not working
    'Account Access',       # login, password, locked account
    'Delivery Issue',       # shipping, tracking, late delivery
    'Refund Request',       # wants money back
    'Product Complaint',    # unhappy with a product
    'Service Outage',       # internet/service not working
    'Feature Request',      # wants a new feature added
    'General Inquiry',      # general questions
    'Cancellation'          # wants to cancel subscription
]

print('Available Tags:')
for i, tag in enumerate(AVAILABLE_TAGS, 1):
    print(f'  {i:2}. {tag}')

Available Tags:
   1. Billing Issue
   2. Technical Problem
   3. Account Access
   4. Delivery Issue
   5. Refund Request
   6. Product Complaint
   7. Service Outage
   8. Feature Request
   9. General Inquiry
  10. Cancellation


In [94]:
# These are realistic customer complaint tickets
# We also store the CORRECT tags so we can measure accuracy later

sample_tickets = [
    {
        'id': 1,
        'ticket': 'I have been charged twice for my subscription this month. Please refund the extra charge immediately.',
        'correct_tags': ['Billing Issue', 'Refund Request']
    },
    {
        'id': 2,
        'ticket': 'My internet connection has been down for the past 3 hours. I work from home and this is very urgent.',
        'correct_tags': ['Service Outage', 'Technical Problem']
    },
    {
        'id': 3,
        'ticket': 'I forgot my password and cannot log into my account. The reset email is not arriving.',
        'correct_tags': ['Account Access']
    },
    {
        'id': 4,
        'ticket': 'My order was supposed to arrive 5 days ago but I still have not received it. The tracking shows it is stuck.',
        'correct_tags': ['Delivery Issue']
    },
    {
        'id': 5,
        'ticket': 'The product I received is completely broken. The screen is cracked and it does not turn on.',
        'correct_tags': ['Product Complaint', 'Refund Request']
    },
    {
        'id': 6,
        'ticket': 'I want to cancel my subscription. I am not satisfied with the service quality.',
        'correct_tags': ['Cancellation']
    },
    {
        'id': 7,
        'ticket': 'It would be great if you could add a dark mode feature to your mobile app.',
        'correct_tags': ['Feature Request']
    },
    {
        'id': 8,
        'ticket': 'What are your business hours? I need to speak with customer support.',
        'correct_tags': ['General Inquiry']
    },
    {
        'id': 9,
        'ticket': 'My app keeps crashing every time I try to open it. I have tried reinstalling but the issue persists.',
        'correct_tags': ['Technical Problem']
    },
    {
        'id': 10,
        'ticket': 'I was billed for a premium plan but I only signed up for the basic plan. Please fix this and refund.',
        'correct_tags': ['Billing Issue', 'Refund Request']
    }
]

print(f'Created {len(sample_tickets)} sample support tickets')
print()
print('Example ticket:')
print(f'  Text: {sample_tickets[0]["ticket"]}')
print(f'  Correct tags: {sample_tickets[0]["correct_tags"]}')

Created 10 sample support tickets

Example ticket:
  Text: I have been charged twice for my subscription this month. Please refund the extra charge immediately.
  Correct tags: ['Billing Issue', 'Refund Request']


In [95]:
def call_llm(prompt, max_tokens=300):
    """
    Sends prompt to Groq API.
    Groq is free and extremely fast — responses in under 1 second!
    Uses Llama3 model which is very powerful.
    """

    payload = {
       'model': 'llama-3.1-8b-instant',    # free powerful model on Groq
        'messages': [
            {'role': 'user', 'content': prompt}
        ],
        'max_tokens': max_tokens,
        'temperature': 0.3           # low = more focused answers
    }

    try:
        response = requests.post(GROQ_URL, headers=HEADERS, json=payload, timeout=30)

        if response.status_code == 200:
            result = response.json()
            # Extract the text response
            return result['choices'][0]['message']['content'].strip()
        else:
            return f'API Error {response.status_code}: {response.text[:200]}'

    except Exception as e:
        return f'Connection error: {str(e)}'


# Test connection
print('Testing Groq connection...')
test = call_llm('Say hello in one sentence.')
print('Response:', test)
print()
if 'Error' not in test:
    print('Groq is working! Ready to tag tickets.')
else:
    print('Something went wrong. Check your API key.')

Testing Groq connection...
Response: Hello, how can I assist you today?

Groq is working! Ready to tag tickets.


In [96]:
def zero_shot_tag(ticket_text):
    """
    Tags a support ticket using Zero-Shot learning.
    No examples provided — just the task description.
    """

    # Build the prompt
    # We list all available tags and ask for top 3
    # This is called 'prompt engineering' — carefully wording our instruction
    prompt = f"""You are a customer support classifier. Your job is to assign tags to support tickets.

Available tags: {', '.join(AVAILABLE_TAGS)}

Support ticket: "{ticket_text}"

Instructions:
- Choose the TOP 3 most relevant tags from the available tags list
- List only the tag names, separated by commas
- Do not explain, just list the tags

Top 3 tags:"""

    # Call the API with this prompt
    response = call_llm(prompt)
    return response


# Test zero-shot on first ticket
print('Testing Zero-Shot on Ticket 1:')
print(f'Ticket: {sample_tickets[0]["ticket"]}')
print(f'Correct tags: {sample_tickets[0]["correct_tags"]}')
print()
result = zero_shot_tag(sample_tickets[0]['ticket'])
print(f'Zero-Shot prediction: {result}')

Testing Zero-Shot on Ticket 1:
Ticket: I have been charged twice for my subscription this month. Please refund the extra charge immediately.
Correct tags: ['Billing Issue', 'Refund Request']

Zero-Shot prediction: Billing Issue, Refund Request, Technical Problem


In [97]:
def few_shot_tag(ticket_text):
    """
    Tags a support ticket using Few-Shot learning.
    3 examples provided before the actual question.
    """

    # These are our FEW-SHOT EXAMPLES
    # We show the AI what good tagging looks like
    # The AI learns the format and logic from these examples
    prompt = f"""You are a customer support classifier. Assign the top 3 tags to each ticket.

Available tags: {', '.join(AVAILABLE_TAGS)}

Here are 3 examples to guide you:

Example 1:
Ticket: "I was charged twice this month on my credit card."
Tags: Billing Issue, Refund Request, General Inquiry

Example 2:
Ticket: "The app crashes every time I open it on my phone."
Tags: Technical Problem, General Inquiry, Feature Request

Example 3:
Ticket: "I cannot log in. My account seems to be locked."
Tags: Account Access, Technical Problem, General Inquiry

Now tag this ticket:
Ticket: "{ticket_text}"
Tags (top 3, comma separated only):"""

    response = call_llm(prompt)
    return response


# Test few-shot on first ticket
print('Testing Few-Shot on Ticket 1:')
print(f'Ticket: {sample_tickets[0]["ticket"]}')
print(f'Correct tags: {sample_tickets[0]["correct_tags"]}')
print()
result = few_shot_tag(sample_tickets[0]['ticket'])
print(f'Few-Shot prediction: {result}')

Testing Few-Shot on Ticket 1:
Ticket: I have been charged twice for my subscription this month. Please refund the extra charge immediately.
Correct tags: ['Billing Issue', 'Refund Request']

Few-Shot prediction: Billing Issue, Refund Request, General Inquiry


In [98]:
def chain_of_thought_tag(ticket_text):
    """
    Tags a support ticket using Chain-of-Thought reasoning.
    Asks the AI to reason step by step before answering.
    """

    prompt = f"""You are a customer support classifier.

Available tags: {', '.join(AVAILABLE_TAGS)}

Support ticket: "{ticket_text}"

Let's think step by step:
Step 1: What is the main problem the customer is facing?
Step 2: What department should handle this?
Step 3: Are there any secondary issues mentioned?
Step 4: Based on my analysis, what are the top 3 most relevant tags?

Please follow these exact steps and end with:
FINAL TAGS: [tag1, tag2, tag3]"""

    response = call_llm(prompt, max_tokens=400)  # more tokens for reasoning
    return response


# Test chain-of-thought on first ticket
print('Testing Chain-of-Thought on Ticket 1:')
print(f'Ticket: {sample_tickets[0]["ticket"]}')
print(f'Correct tags: {sample_tickets[0]["correct_tags"]}')
print()
result = chain_of_thought_tag(sample_tickets[0]['ticket'])
print(f'Chain-of-Thought prediction:')
print(result)

Testing Chain-of-Thought on Ticket 1:
Ticket: I have been charged twice for my subscription this month. Please refund the extra charge immediately.
Correct tags: ['Billing Issue', 'Refund Request']

Chain-of-Thought prediction:
Step 1: What is the main problem the customer is facing?
The main problem the customer is facing is that they have been charged twice for their subscription this month, and they are requesting a refund for the extra charge.

Step 2: What department should handle this?
Based on the issue, the billing department should handle this as it involves a charge and a refund request.

Step 3: Are there any secondary issues mentioned?
There are no secondary issues mentioned in the ticket. The customer is specifically asking for a refund for the extra charge.

Step 4: Based on my analysis, what are the top 3 most relevant tags?
Based on the issue, the top 3 most relevant tags are:
- Billing Issue: The customer is facing a billing issue with being charged twice.
- Refund Req

In [99]:
# Run all 3 methods on all 10 tickets and collect results
# This will take a few minutes due to API calls

print('Running all 3 methods on all tickets...')
print('This may take 3-5 minutes due to API calls. Please wait...')
print()

results = []

for ticket in sample_tickets:
    print(f'Processing Ticket {ticket["id"]}...')

    # Get predictions from all 3 methods
    zero_result = zero_shot_tag(ticket['ticket'])
    time.sleep(1)  # wait 1 second between calls to avoid hitting API rate limit

    few_result = few_shot_tag(ticket['ticket'])
    time.sleep(1)

    cot_result = chain_of_thought_tag(ticket['ticket'])
    time.sleep(1)

    # Store all results for this ticket
    results.append({
        'Ticket ID': ticket['id'],
        'Ticket Text': ticket['ticket'][:60] + '...',  # breaking for display
        'Correct Tags': ', '.join(ticket['correct_tags']),
        'Zero-Shot': zero_result,
        'Few-Shot': few_result,
        'Chain-of-Thought': cot_result
    })

print()
print('All tickets processed!')

# Convert results to a DataFrame for easy viewing
results_df = pd.DataFrame(results)
print(results_df[['Ticket ID', 'Correct Tags', 'Zero-Shot', 'Few-Shot']].to_string())

Running all 3 methods on all tickets...
This may take 3-5 minutes due to API calls. Please wait...

Processing Ticket 1...
Processing Ticket 2...
Processing Ticket 3...
Processing Ticket 4...
Processing Ticket 5...
Processing Ticket 6...
Processing Ticket 7...
Processing Ticket 8...
Processing Ticket 9...
Processing Ticket 10...

All tickets processed!
   Ticket ID                       Correct Tags                                              Zero-Shot                                                                                                                                                                                                                 Few-Shot
0          1      Billing Issue, Refund Request       Billing Issue, Refund Request, Technical Problem                                                                                                                                                                           Billing Issue, Refund Request, General Inquiry
1    